In [6]:
storage_account = "straccajs"
processed_path = f"abfss://processed@{storage_account}.dfs.core.windows.net/yellow/"
processed_clean_path = f"abfss://processed@{storage_account}.dfs.core.windows.net/yellow_clean/"
rejected_path = f"abfss://rejected@{storage_account}.dfs.core.windows.net/yellow/"

StatementMeta(sparkpool1, 1, 2, Finished, Available, Finished, False)

In [7]:
# CELL-2 Read processed data and flag invalid rows, Loads January data from /processed/yellow/, counts input rows, and creates reject_reason labels for rows that fail quality rules.
#This supports validation and bad-record quarantine.
from pyspark.sql import functions as F

df = spark.read.parquet(processed_path).cache()
total_in = df.count()
print(f"Starting rows: {total_in:,}")

df_flagged = df.withColumn(
    "reject_reason",
    F.when(F.col("tpep_pickup_datetime").isNull(), "null_pickup")
     .when(F.col("tpep_dropoff_datetime").isNull(), "null_dropoff")
     .when(F.col("tpep_dropoff_datetime") <= F.col("tpep_pickup_datetime"), "dropoff_before_pickup")
     .when(F.col("trip_distance") <= 0, "non_positive_distance")
     .when(F.col("fare_amount") < 0, "negative_fare")
     .when(F.col("passenger_count").isNull() | (F.col("passenger_count") == 0), "invalid_passenger_count")
     .when(F.year("tpep_pickup_datetime") != 2024, "wrong_year")
     .otherwise(None)
).cache()

StatementMeta(sparkpool1, 1, 3, Finished, Available, Finished, False)

Starting rows: 2,964,624


In [8]:
# CELL 3 - Split good and bad rows, then show reject breakdown
# Separates quarantined rows from valid rows and prints counts by reject_reason.
# This shows how much data is rejected and why.

df_bad = df_flagged.filter(F.col("reject_reason").isNotNull())
df_good = df_flagged.filter(F.col("reject_reason").isNull()).drop("reject_reason")

print("\nReject reason breakdown:")
df_bad.groupBy("reject_reason").count().orderBy(F.desc("count")).show(truncate=False)

bad_count = df_bad.count()
good_count = total_in - bad_count
print(f"Quarantined: {bad_count:,} ({bad_count/total_in*100:.2f}%)")
print(f"Kept:        {good_count:,}")

StatementMeta(sparkpool1, 1, 4, Finished, Available, Finished, False)


Reject reason breakdown:
+-----------------------+------+
|reject_reason          |count |
+-----------------------+------+
|invalid_passenger_count|145916|
|non_positive_distance  |59613 |
|negative_fare          |34065 |
|dropoff_before_pickup  |870   |
|wrong_year             |14    |
+-----------------------+------+

Quarantined: 240,478 (8.11%)
Kept:        2,724,146


In [9]:
# CELL 4 - Remove duplicate rows defensively
# Drops duplicate trips using key trip fields and reports how many were removed.
# This is a defensive cleaning step before feature engineering.

df_dedup = df_good.dropDuplicates(
    ["tpep_pickup_datetime", "tpep_dropoff_datetime", "PULocationID", "DOLocationID", "fare_amount"]
)
dedup_count = df_dedup.count()
print(f"After dedup: {dedup_count:,} ({good_count - dedup_count:,} duplicates removed)")

StatementMeta(sparkpool1, 1, 5, Finished, Available, Finished, False)

After dedup: 2,724,063 (83 duplicates removed)


In [10]:
# CELL 5 - Cast key fields and create derived features
# Converts selected columns to cleaner types and engineers useful features
# for later analysis, such as trip duration, pickup hour, weekday, weekend,and pickup month.

df_clean = (df_dedup
    .withColumn("VendorID", F.col("VendorID").cast("int"))
    .withColumn("passenger_count", F.col("passenger_count").cast("int"))
    .withColumn("trip_duration_min",
                (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime"))/60)
    .withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
    .withColumn("pickup_day", F.dayofweek("tpep_pickup_datetime"))
    .withColumn("is_weekend", F.col("pickup_day").isin([1, 7]))
    .withColumn("pickup_month", F.month("tpep_pickup_datetime"))
)

StatementMeta(sparkpool1, 1, 6, Finished, Available, Finished, False)

In [11]:
# CELL 6 - Write rejected and cleaned outputs
# Saves quarantined bad rows to /rejected/yellow/ and cleaned rows to /processed/yellow_clean/. This completes the cleaning stage.

df_bad.write.mode("overwrite").parquet(rejected_path)
df_clean.write.mode("overwrite").parquet(processed_clean_path)
print(f"\nDone. Clean: {dedup_count:,}, Rejected: {bad_count:,}")

df.unpersist()
df_flagged.unpersist()

StatementMeta(sparkpool1, 1, 7, Finished, Available, Finished, False)


Done. Clean: 2,724,063, Rejected: 240,478


DataFrame[VendorID: int, tpep_pickup_datetime: timestamp_ntz, tpep_dropoff_datetime: timestamp_ntz, passenger_count: bigint, trip_distance: double, RatecodeID: bigint, store_and_fwd_flag: string, PULocationID: int, DOLocationID: int, payment_type: bigint, fare_amount: double, extra: double, mta_tax: double, tip_amount: double, tolls_amount: double, improvement_surcharge: double, total_amount: double, congestion_surcharge: double, Airport_fee: double, reject_reason: string]